# Deploying Your First AI App

Ship an LLM-powered API and UI to production with FastAPI, Streamlit, Docker, and proper config management.

**Topics:** FastAPI + LLM, Pydantic models, Streamlit chatbot UI, environment variables, Docker, health checks, rate limiting

## 1. FastAPI Endpoint Wrapping an LLM Call

In [ ]:
# app/main.py — production FastAPI app
# Run with: uvicorn app.main:app --reload

FASTAPI_APP = '''
from fastapi import FastAPI, HTTPException, Depends
from fastapi.middleware.cors import CORSMiddleware
from openai import OpenAI, RateLimitError
import os

from .models import ChatRequest, ChatResponse
from .config import Settings, get_settings

app = FastAPI(
    title="AI Chat API",
    version="1.0.0",
    description="Production-ready LLM chat endpoint",
)

app.add_middleware(
    CORSMiddleware,
    allow_origins=["*"],  # restrict in production
    allow_methods=["POST", "GET"],
    allow_headers=["*"],
)

@app.get("/health")
async def health():
    """Health check endpoint — used by load balancers and K8s."""
    return {"status": "ok", "version": app.version}

@app.post("/chat", response_model=ChatResponse)
async def chat(
    request: ChatRequest,
    settings: Settings = Depends(get_settings),
):
    client = OpenAI(api_key=settings.openai_api_key)
    try:
        response = client.chat.completions.create(
            model=settings.llm_model,
            messages=[
                {"role": "system", "content": settings.system_prompt},
                *[m.dict() for m in request.messages],
            ],
            max_tokens=settings.max_tokens,
            temperature=request.temperature,
        )
        return ChatResponse(
            content=response.choices[0].message.content,
            model=response.model,
            usage={
                "prompt_tokens": response.usage.prompt_tokens,
                "completion_tokens": response.usage.completion_tokens,
            },
        )
    except RateLimitError:
        raise HTTPException(status_code=429, detail="Rate limit exceeded. Try again shortly.")
    except Exception as e:
        raise HTTPException(status_code=500, detail=str(e))
'''

print('FastAPI app structure:')
print('  GET  /health      → liveness check')
print('  POST /chat        → LLM chat completion')
print()
print('Key patterns:')
print('  - Depends(get_settings): injects config from env vars')
print('  - response_model=ChatResponse: auto validates + docs output')
print('  - RateLimitError → 429: proper HTTP status mapping')
print(FASTAPI_APP[:200] + '...')

## 2. Request/Response Models with Pydantic

In [ ]:
from pydantic import BaseModel, Field, field_validator
from typing import Optional

class Message(BaseModel):
    role: str = Field(..., pattern='^(user|assistant|system)$')
    content: str = Field(..., min_length=1, max_length=10000)

class ChatRequest(BaseModel):
    messages: list[Message] = Field(..., min_length=1, max_length=50)
    temperature: float = Field(default=0.7, ge=0.0, le=2.0)
    stream: bool = False
    
    @field_validator('messages')
    @classmethod
    def last_message_must_be_user(cls, messages):
        if messages[-1].role != 'user':
            raise ValueError('Last message must be from user')
        return messages

class ChatResponse(BaseModel):
    content: str
    model: str
    usage: dict[str, int]
    
    @property
    def total_tokens(self) -> int:
        return self.usage.get('prompt_tokens', 0) + self.usage.get('completion_tokens', 0)

class Settings(BaseModel):
    """App config loaded from environment variables."""
    openai_api_key: str
    llm_model: str = 'gpt-4o-mini'
    max_tokens: int = 1024
    system_prompt: str = 'You are a helpful assistant.'

# Validation demo
import json

valid_request = ChatRequest(
    messages=[
        Message(role='user', content='Hello, what can you do?')
    ],
    temperature=0.7,
)
print('Valid request:')
print(json.dumps(valid_request.model_dump(), indent=2))

# Show validation error
try:
    bad = ChatRequest(messages=[Message(role='assistant', content='Hi')], temperature=3.0)
except Exception as e:
    print(f'\nValidation caught: {type(e).__name__}')
    print('  temperature must be <= 2.0')
    print('  last message must be from user')

## 3. Streamlit Chatbot UI

In [ ]:
# streamlit_app.py — run with: streamlit run streamlit_app.py

STREAMLIT_APP = '''
import streamlit as st
from openai import OpenAI
import os

st.set_page_config(page_title="AI Chat", page_icon="🤖", layout="centered")
st.title("AI Assistant")

# Initialize session state (persists across reruns)
if "messages" not in st.session_state:
    st.session_state.messages = []
if "client" not in st.session_state:
    st.session_state.client = OpenAI(api_key=os.getenv("OPENAI_API_KEY"))

# Sidebar config
with st.sidebar:
    st.header("Settings")
    model = st.selectbox("Model", ["gpt-4o-mini", "gpt-4o"], index=0)
    temperature = st.slider("Temperature", 0.0, 2.0, 0.7, 0.1)
    if st.button("Clear conversation"):
        st.session_state.messages = []
        st.rerun()

# Display conversation history
for msg in st.session_state.messages:
    with st.chat_message(msg["role"]):
        st.markdown(msg["content"])

# Handle new input
if prompt := st.chat_input("Ask anything..."):
    # Display user message
    st.session_state.messages.append({"role": "user", "content": prompt})
    with st.chat_message("user"):
        st.markdown(prompt)
    
    # Stream assistant response
    with st.chat_message("assistant"):
        stream = st.session_state.client.chat.completions.create(
            model=model,
            messages=st.session_state.messages,
            temperature=temperature,
            stream=True,
        )
        response = st.write_stream(stream)  # streams tokens to UI
    
    st.session_state.messages.append({"role": "assistant", "content": response})
'''

print('Streamlit chatbot — key concepts:')
print('  st.session_state: persists data across UI reruns')
print('  st.chat_message(): renders chat bubbles')
print('  st.chat_input(): chat input box at bottom')
print('  st.write_stream(): streams tokens to UI in real-time')
print()
print('Run: streamlit run streamlit_app.py')
print('Opens at: http://localhost:8501')

## 4. Environment Variable Management

In [ ]:
from pydantic_settings import BaseSettings
from functools import lru_cache

class AppSettings(BaseSettings):
    """Reads config from environment variables or .env file."""
    
    # Required
    openai_api_key: str
    
    # Optional with defaults
    llm_model: str = 'gpt-4o-mini'
    max_tokens: int = 1024
    system_prompt: str = 'You are a helpful assistant.'
    log_level: str = 'INFO'
    
    class Config:
        env_file = '.env'           # reads from .env in dev
        env_file_encoding = 'utf-8'
        case_sensitive = False      # OPENAI_API_KEY == openai_api_key

@lru_cache()  # cache: only reads env once per process
def get_settings() -> AppSettings:
    return AppSettings()

# .env file content
dot_env_content = """
# .env — NEVER commit this file!
OPENAI_API_KEY=sk-...
LLM_MODEL=gpt-4o-mini
MAX_TOKENS=1024
LOG_LEVEL=INFO
"""

gitignore_additions = """
# Always in .gitignore:
.env
.env.local
.env.*.local
*.pem
secrets/
"""

print('.env file example:')
print(dot_env_content)
print('Always add to .gitignore:')
print(gitignore_additions)
print('In production: use AWS Secrets Manager, GCP Secret Manager, or Vault')
print('Never hardcode API keys in source code.')

## 5. Dockerfile for AI Applications

In [ ]:
dockerfile = '''
# Dockerfile — multi-stage for smaller final image
FROM python:3.11-slim AS builder

WORKDIR /app

# Install dependencies first (cached layer if requirements.txt unchanged)
COPY requirements.txt .
RUN pip install --no-cache-dir --user -r requirements.txt

# ── Production stage ─────────────────────────────────────────
FROM python:3.11-slim

WORKDIR /app

# Create non-root user (security best practice)
RUN useradd -m -u 1000 appuser

# Copy installed packages from builder
COPY --from=builder /root/.local /home/appuser/.local

# Copy application code
COPY --chown=appuser:appuser ./app ./app

USER appuser
ENV PATH=/home/appuser/.local/bin:$PATH

EXPOSE 8000

# Health check — Docker restarts container if this fails
HEALTHCHECK --interval=30s --timeout=10s --retries=3 \\
    CMD curl -f http://localhost:8000/health || exit 1

CMD ["uvicorn", "app.main:app", "--host", "0.0.0.0", "--port", "8000", "--workers", "2"]
'''

docker_compose = '''
# docker-compose.yml
services:
  api:
    build: .
    ports:
      - "8000:8000"
    env_file: .env        # inject secrets from .env
    restart: unless-stopped
    healthcheck:
      test: ["CMD", "curl", "-f", "http://localhost:8000/health"]
      interval: 30s
      timeout: 10s
      retries: 3
'''

print('Dockerfile key practices:')
practices = [
    'Multi-stage build: separates build deps from runtime (smaller image)',
    'Non-root user: security — never run as root in production',
    'COPY requirements first: Docker caches this layer',
    'HEALTHCHECK: Docker/K8s restarts unhealthy containers',
    '--no-cache-dir pip: smaller image, no stale cache',
]
for p in practices:
    print(f'  ✓ {p}')
print()
print('Commands:')
print('  docker build -t ai-app:latest .')
print('  docker run -p 8000:8000 --env-file .env ai-app:latest')
print('  docker compose up -d')

## Exercises

1. **Rate limiter middleware:** Add a FastAPI middleware that limits each IP address to 10 requests per minute using an in-memory counter with a sliding window. Return HTTP 429 when exceeded.

2. **Streaming FastAPI:** Modify the `/chat` endpoint to support server-sent events (SSE) streaming when `request.stream=True`, using FastAPI's `StreamingResponse` and OpenAI's streaming API.

3. **Multi-model Streamlit:** Extend the Streamlit app to let users choose between OpenAI and Anthropic Claude models, handle the different API response formats, and display cost estimates after each response.